# Unüberwachtes Lernen für AML: PCA & Clustering

Dieses Notebook demonstriert **PCA** zur Dimensionsreduktion und **k‑Means Clustering** zur Gruppierung von Transaktionen. Ziel ist die Identifikation von Ausreißern und ungewöhnlichen Mustern.

**Lernziele:**
- PCA auf Transaktionsdaten anwenden und interpretieren
- k‑Means Clustering durchführen und Elbow‑Methode nutzen
- Anomalien als Punkte mit großer Distanz zum nächsten Zentroid erkennen

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Synthetische Transaktionsdaten mit 5 Merkmalen
np.random.seed(42)
n = 500
amount = np.random.gamma(2, 500, n)
amount_log = np.log1p(amount)
hour = np.random.randint(0, 24, n)
weekend = np.random.choice([0, 1], n, p=[0.7, 0.3])
transaction_count = np.random.poisson(5, n)  # Anzahl Transaktionen pro Kunde (simuliert)
location_change = np.random.exponential(10, n)  # Distanz zum letzten Standort

df = pd.DataFrame({
    'amount': amount,
    'amount_log': amount_log,
    'hour': hour,
    'weekend': weekend,
    'tx_count': transaction_count,
    'location_change': location_change
})

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['amount_log', 'hour', 'weekend', 'tx_count', 'location_change']])

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print("Erklärte Varianz pro Komponente:", pca.explained_variance_ratio_)

plt.scatter(X_pca[:,0], X_pca[:,1], alpha=0.6)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA der Transaktionsdaten')
plt.show()

## k‑Means Clustering

In [ ]:
inertias = []
sil_scores = []
K_range = range(2, 10)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_scaled, kmeans.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(K_range, inertias, marker='o')
ax1.set_xlabel('Anzahl Cluster k')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Methode')
ax2.plot(K_range, sil_scores, marker='o')
ax2.set_xlabel('Anzahl Cluster k')
ax2.set_ylabel('Silhouetten Score')
ax2.set_title('Silhouetten Score')
plt.show()

In [1]:
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
df['cluster'] = clusters

# PCA‑Plot mit Clustern
plt.scatter(X_pca[:,0], X_pca[:,1], c=clusters, cmap='viridis', alpha=0.6)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title(f'k‑Means Cluster (k={optimal_k})')
plt.colorbar(label='Cluster')
plt.show()

# Anomalieerkennung: Distanz zum nächsten Zentroid
distances = kmeans.transform(X_scaled).min(axis=1)
threshold = np.percentile(distances, 95)
anomalies = df[distances > threshold]
print(f"Anzahl Anomalien (95. Perzentil): {len(anomalies)}")
print(anomalies[['amount', 'tx_count', 'location_change']].head())

NameError: name 'KMeans' is not defined

## Diskussion

PCA reduziert die Dimensionen und zeigt Clusterstrukturen. Die Elbow‑Methode und der Silhouetten Score helfen bei der Wahl von k. Transaktionen, die weit von ihrem Clusterzentrum entfernt sind, können als Anomalien betrachtet werden – ein einfacher Ansatz zur Betrugserkennung ohne gelabelte Daten.

**Verbesserungshinweis:** Für echte AML‑Daten sollten Sie die Cluster regelmäßig neu berechnen und die Schwellwerte anpassen.